<a href="https://colab.research.google.com/github/GokulM8/Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Setup — connect to the warehouse (DuckDB, remote)

Same lane as ML-04: **Content Refresh / Opportunity Scoring**, same table (`fact_content_daily_performance`), March 2026 as the mid-panel month.

In [2]:
%pip install -q duckdb huggingface_hub

In [3]:
from google.colab import userdata
from huggingface_hub import HfApi
import duckdb
import numpy as np
import pandas as pd
import os

HF_TOKEN = userdata.get("HF_TOKEN")

In [4]:
api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
config_files = sorted(f for f in all_files if "fact_content_daily_performance" in f and f.endswith(".parquet"))
print(f"Found {len(config_files)} parquet file(s) for this config.")

Found 19 parquet file(s) for this config.


In [5]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

con.sql(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

remote_paths = [f"hf://datasets/FlyRank/internship-warehouse/{f}" for f in config_files]
paths_sql = "[" + ", ".join(f"'{p}'" for p in remote_paths) + "]"
con.sql(f"CREATE OR REPLACE VIEW fact AS SELECT * FROM read_parquet({paths_sql})")

In [6]:
# Same monthly, content-level feature frame as w03_data_contract — March 2026 only
features = con.sql("""
    WITH march AS (
        SELECT *, CAST(strftime(report_date, '%d') AS INTEGER) AS day
        FROM fact
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    )
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_impressions) AS total_impressions,
        SUM(CASE WHEN day <= 15 THEN gsc_clicks ELSE 0 END) AS front_half_clicks,
        SUM(CASE WHEN day > 15 THEN gsc_clicks ELSE 0 END) AS back_half_clicks
    FROM march
    GROUP BY client_hash_id, content_hash_id
""").df().fillna(0)

print("Feature frame shape:", features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 7)


,client_hash_id,content_hash_id,avg_ctr,avg_position,total_impressions,front_half_clicks,back_half_clicks
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,0.000000,5.147402,181.0,0.0,0.0
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,0.021739,4.828125,46.0,1.0,0.0
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,0.001112,5.145765,899.0,1.0,0.0
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,0.000000,4.909314,34.0,0.0,0.0
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,0.000000,6.969536,3108.0,0.0,0.0


### Signal 1 — staleness / decline trend (behind the refresh flag)

**Hypothesis:** a page whose clicks dropped within the month (back half lower than front half — our staleness proxy) should also be losing search rank, on average, since that's the mechanism the refresh flag leans on. Bucket by declining vs. not, show `n` and mean `avg_position` per bucket.

In [7]:
features["is_declining"] = (features["back_half_clicks"] < features["front_half_clicks"]).astype(int)

bucket1 = (
    features.groupby("is_declining")
    .agg(n=("content_hash_id", "count"), avg_position=("avg_position", "mean"))
    .reset_index()
)
bucket1

,is_declining,n,avg_position
0,0,302448,8.267999
1,1,28989,11.281538


In [ ]:
# VERDICT — signal 1
pos_declining = bucket1.loc[bucket1.is_declining == 1, "avg_position"].values[0]
pos_stable = bucket1.loc[bucket1.is_declining == 0, "avg_position"].values[0]
n_declining = int(bucket1.loc[bucket1.is_declining == 1, "n"].values[0])
n_stable = int(bucket1.loc[bucket1.is_declining == 0, "n"].values[0])

diff = pos_declining - pos_stable  # positive = declining pages rank worse, as expected
POSITION_MARGIN = 1.0

if diff > POSITION_MARGIN:
    verdict1 = "CONFIRMED"
elif diff < -POSITION_MARGIN:
    verdict1 = "OPPOSITE"
else:
    verdict1 = "FALSE"

print(f"n(declining)={n_declining}, n(stable)={n_stable}")
print(f"avg_position declining={pos_declining:.2f} vs stable={pos_stable:.2f} (diff={diff:+.2f})")
print("VERDICT:", verdict1)
if min(n_declining, n_stable) < 30:
    print("Caution: one bucket has under 30 pages — treat this verdict as provisional.")

### Signal 2 — CTR vs. position (behind the CTR-fix logic)

**Hypothesis:** CTR should fall as position gets worse — that's the assumption the CTR-fix flag leans on (a page ranking well but getting low CTR is the real anomaly worth fixing). Bucket by position range, show `n` and mean `avg_ctr` per bucket.

In [8]:
def position_bucket(p):
    if p <= 3:
        return "1-3"
    elif p <= 10:
        return "4-10"
    elif p <= 20:
        return "11-20"
    else:
        return "21+"

features["position_bucket"] = features["avg_position"].apply(position_bucket)

bucket2 = (
    features.groupby("position_bucket")
    .agg(n=("content_hash_id", "count"), avg_ctr=("avg_ctr", "mean"))
    .reindex(["1-3", "4-10", "11-20", "21+"])
    .reset_index()
)
bucket2

,position_bucket,n,avg_ctr
0,1-3,172277,0.001265
1,4-10,81988,0.004926
2,11-20,32203,0.003211
3,21+,44969,0.001928


In [9]:
# VERDICT — signal 2
ctr_values = bucket2["avg_ctr"].tolist()
is_monotonic_decreasing = all(ctr_values[i] >= ctr_values[i + 1] for i in range(len(ctr_values) - 1))
is_monotonic_increasing = all(ctr_values[i] <= ctr_values[i + 1] for i in range(len(ctr_values) - 1))
spread = max(ctr_values) - min(ctr_values)

if is_monotonic_decreasing and spread > 0.01:
    verdict2 = "CONFIRMED"
elif is_monotonic_increasing and spread > 0.01:
    verdict2 = "OPPOSITE"
elif spread <= 0.005:
    verdict2 = "FALSE"
else:
    verdict2 = "MIXED"

print(bucket2)
print("VERDICT:", verdict2)

  position_bucket       n   avg_ctr
0             1-3  172277  0.001265
1            4-10   81988  0.004926
2           11-20   32203  0.003211
3             21+   44969  0.001928
VERDICT: FALSE


### My rule, in plain words

**The rule:** flag a page for refresh when it shows a real within-month decline in clicks (front half of March higher than back half) *and* it has meaningful search volume — a decline on a page almost nobody sees isn't worth acting on. The score ranks by how much organic traffic is actually at stake, not by decline percentage alone (which is noisy on tiny numbers).

**Score:** `score = max(0, front_half_clicks - back_half_clicks) * log1p(total_impressions)` — decline magnitude weighted by scale, so a real drop on a high-traffic page ranks above a noisy blip on a page nobody visits.

**Reason code (one, applied to every ranked row):** `DECLINING_TRAFFIC_HIGH_VOLUME` — the row is on the queue because of this pattern; the score says how strongly.

**Action label (one, applied to every ranked row):** `REVIEW_FOR_REFRESH` — the rule only proposes the queue order; a human makes the final call per page in the Top-10 review below.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
features["decline_magnitude"] = (features["front_half_clicks"] - features["back_half_clicks"]).clip(lower=0)
features["score"] = features["decline_magnitude"] * np.log1p(features["total_impressions"])
features["reason_code"] = "DECLINING_TRAFFIC_HIGH_VOLUME"
features["action_label"] = "REVIEW_FOR_REFRESH"

ranked = features.sort_values("score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

os.makedirs("work/outputs", exist_ok=True)
output_cols = [
    "rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action_label",
    "front_half_clicks", "back_half_clicks", "total_impressions", "avg_position", "avg_ctr",
]
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(ranked))
ranked[output_cols].head(10)

Rows written: 331437


,rank,client_hash_id,content_hash_id,score,reason_code,action_label,front_half_clicks,back_half_clicks,total_impressions,avg_position,avg_ctr
0,1,client_20259bd6705d81d4,content_0ec90963d98b97a5,3556.923996,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,914.0,612.0,130338.0,3.249691,0.011708
1,2,client_20259bd6705d81d4,content_0175875757a5a1b3,2687.449340,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,328.0,51.0,16349.0,6.051372,0.023182
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,2556.489553,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,843.0,637.0,245276.0,2.854514,0.006034
3,4,client_e547b89c05043229,content_8d7d99f109e19aa2,2139.097005,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,232.0,57.0,203497.0,2.563756,0.001420
4,5,client_20259bd6705d81d4,content_623f10411a2328a8,1952.841707,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,264.0,73.0,27564.0,5.212509,0.012226
5,6,client_0fa64a184f18a4a0,content_5ebc94f67db6f51c,1679.216651,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,300.0,132.0,21923.0,2.702855,0.019705
6,7,client_e547b89c05043229,content_c9a0c2fdbdbfb562,1541.868647,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,439.0,300.0,65681.0,2.446912,0.011251
7,8,client_62f4a7e64f5e0096,content_7172a7fad43f0998,1517.138818,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,493.0,369.0,205867.0,3.367835,0.004187
8,9,client_23a62021009f63c4,content_ce467ecc5defe43f,1419.502034,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,199.0,55.0,19103.0,11.125163,0.013296
9,10,client_20259bd6705d81d4,content_ef471db6287ccd4b,1373.820662,DECLINING_TRAFFIC_HIGH_VOLUME,REVIEW_FOR_REFRESH,281.0,152.0,42182.0,6.326826,0.010265


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
top10 = ranked.head(10).copy()

for _, row in top10.iterrows():
    decline_pct = (
        row["decline_magnitude"] / row["front_half_clicks"] * 100
        if row["front_half_clicks"] > 0 else float("nan")
    )
    print(f"Rank {int(row['rank'])} — content {row['content_hash_id']} (client {row['client_hash_id']})")
    print(f"  Action: {row['action_label']}  |  Reason: {row['reason_code']}")
    print(
        f"  Why it's here: clicks dropped from {int(row['front_half_clicks'])} (first half) to "
        f"{int(row['back_half_clicks'])} (second half) — a {decline_pct:.0f}% drop — on "
        f"{int(row['total_impressions'])} total March impressions (real, not tiny, volume)."
    )
    print(
        "  What would make this wrong: the split could be catching one anomalous day rather than a "
        "real trend; the page may already have been refreshed since March; or the dip could be "
        "seasonal or site-wide rather than specific to this page."
    )
    print()

Rank 1 — content content_0ec90963d98b97a5 (client client_20259bd6705d81d4)
  Action: REVIEW_FOR_REFRESH  |  Reason: DECLINING_TRAFFIC_HIGH_VOLUME
  Why it's here: clicks dropped from 914 (first half) to 612 (second half) — a 33% drop — on 130338 total March impressions (real, not tiny, volume).
  What would make this wrong: the split could be catching one anomalous day rather than a real trend; the page may already have been refreshed since March; or the dip could be seasonal or site-wide rather than specific to this page.

Rank 2 — content content_0175875757a5a1b3 (client client_20259bd6705d81d4)
  Action: REVIEW_FOR_REFRESH  |  Reason: DECLINING_TRAFFIC_HIGH_VOLUME
  Why it's here: clicks dropped from 328 (first half) to 51 (second half) — a 84% drop — on 16349 total March impressions (real, not tiny, volume).
  What would make this wrong: the split could be catching one anomalous day rather than a real trend; the page may already have been refreshed since March; or the dip could be 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# Weak picks: rows where the "decline" is really just noise on a tiny base
weak_picks = top10[top10["front_half_clicks"] < 20]
print("Weak picks in the top 10 (front_half_clicks < 20 — a single-day swing can look like a trend):")
weak_picks[["rank", "content_hash_id", "front_half_clicks", "back_half_clicks", "total_impressions"]]

Weak picks in the top 10 (front_half_clicks < 20 — a single-day swing can look like a trend):


,rank,content_hash_id,front_half_clicks,back_half_clicks,total_impressions


In [13]:
# Leakage check
score_inputs = ["front_half_clicks", "back_half_clicks", "total_impressions"]
print("Columns feeding the score:", score_inputs)
print("All three are pulled from the March 2026 slice only (day 1-31) — no April/June rows touched,")
print("and fact_content_daily_performance has no pre-computed product/opportunity flag columns to leak from.")

Columns feeding the score: ['front_half_clicks', 'back_half_clicks', 'total_impressions']
All three are pulled from the March 2026 slice only (day 1-31) — no April/June rows touched,
and fact_content_daily_performance has no pre-computed product/opportunity flag columns to leak from.


## Self-check

Before you submit, confirm each line honestly:

- [-] Every section above is filled — markdown thinking AND the code that backs it
- [-] The notebook runs top to bottom with no errors (Runtime → Run all)
- [-] No client names, URLs, or private queries anywhere
- [-] My claims use careful words: observed, measured, directional, decision-support
- [-] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.